# Facial Emotion Recognition on FER-2013

Classify 48x48 grayscale face images into 7 emotions: angry, disgust, fear, happy, neutral, sad and surprise. This notebook rebuilds my original CNN as a baseline, then improves it, then looks closely at where it still fails.

Run `python download_data.py` first to fetch the dataset into `data/`.

In [ ]:
import io, os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")  # any Keras 3 backend works

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import keras
from keras import layers
from sklearn.model_selection import train_test_split

EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
keras.utils.set_random_seed(42)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Load the data

FER-2013 comes with a 28,709-image training set and a 7,178-image test set. I hold out 10% of the training images (stratified) for validation, so the test set is only used once at the end.

In [ ]:
def load(split):
    df = pd.read_parquet(f"data/{split}.parquet")
    x = np.stack([np.asarray(Image.open(io.BytesIO(b)).convert("L")) for b in df.img_bytes])[..., None]
    return x, df.labels.to_numpy().astype("int64")

x_all, y_all = load("train")
x_test, y_test = load("test")
x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=0.1, stratify=y_all, random_state=42)
print(f"train {x_train.shape}  val {x_val.shape}  test {x_test.shape}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1.4]})
counts = np.bincount(y_all, minlength=7)
ax1.barh(EMOTIONS, counts, color="#2a78d6")
for i, c in enumerate(counts):
    ax1.text(c + 80, i, f"{c:,}", va="center", fontsize=9)
ax1.invert_yaxis(); ax1.set_title("Training images per class"); ax1.set_xlim(0, counts.max() * 1.15)
ax2.axis("off")
rng = np.random.default_rng(0)
grid = np.vstack([np.hstack([x_all[rng.choice(np.where(y_all == k)[0])][..., 0] for _ in range(6)]) for k in range(7)])
ax2.imshow(grid, cmap="gray")
for k, name in enumerate(EMOTIONS):
    ax2.text(-6, k * 48 + 24, name, ha="right", va="center", fontsize=9)
ax2.set_title("Random examples")
plt.tight_layout(); plt.show()

The classes are very unbalanced: **disgust** has only 436 images, 16 times fewer than **happy**. The faces vary in pose, lighting and age, some are cartoons or partly covered, and the labels themselves are noisy (human agreement on FER-2013 is estimated at about 65%).

## 2. Baseline: my original CNN

Four convolution layers, two dropout layers and a 1,024-unit dense layer, trained for 10 epochs with Adam, as in my first version of this project.

In [ ]:
def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        ax1.plot(h.history["accuracy"], label=f"{name} train")
        ax1.plot(h.history["val_accuracy"], "--", label=f"{name} validation")
        ax2.plot(h.history["loss"], label=f"{name} train")
        ax2.plot(h.history["val_loss"], "--", label=f"{name} validation")
    for ax, title in ((ax1, "Accuracy"), (ax2, "Loss")):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.show()

baseline = keras.Sequential([
    keras.Input((48, 48, 1)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(1024, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7, activation="softmax"),
], name="baseline")
baseline.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
baseline.summary()

In [ ]:
hist_base = baseline.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=10, batch_size=64, verbose=2)

In [ ]:
plot_history({"baseline": hist_base})
base_loss, base_acc = baseline.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test accuracy: {base_acc:.1%}")

## 3. A better CNN

The baseline is still improving when it stops, and its training accuracy runs ahead of validation. Changes:

- **Four VGG-style blocks** (two 3x3 convolutions each, 64 → 512 filters) to learn richer features
- **Batch normalisation** after every convolution for faster, more stable training
- **Data augmentation**: horizontal flips, small rotations, zooms and shifts (a smiling face flipped is still smiling)
- **Global average pooling** instead of a huge `Flatten` → `Dense` layer
- **Learning-rate schedule and early stopping** on the validation set, so training runs as long as it helps

In [ ]:
def conv_block(x, filters):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(0.25)(x)

inputs = keras.Input((48, 48, 1))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.08)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomTranslation(0.1, 0.1)(x)
x = layers.Rescaling(1 / 255)(x)
for filters in (64, 128, 256, 512):
    x = conv_block(x, filters)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(7, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="emotion_cnn")
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Parameters: {model.count_params():,}")

In [ ]:
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
]
hist = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=50, batch_size=64,
                 callbacks=callbacks, verbose=2)

In [ ]:
plot_history({"baseline": hist_base, "improved": hist})
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test accuracy: {base_acc:.1%}")
print(f"Improved test accuracy: {test_acc:.1%}  ({len(hist.history['loss'])} epochs)")